# 8주차 중간고사 — PDF 문서 RAG 챗봇 만들기
### 실습 주제: 인공지능 RAG(검색 증강 생성) 챗봇으로 「2026 복지사업 가이드북」 PDF 질의응답하기

- 시간 120분 / 총점 100점 / 개인 평가 / API 키 불필요
- 7주차 워크북 1·2와 같은 방법으로 **2026 복지사업 가이드북**(근로복지공단, 28쪽)을 검색합니다.
- **답안은 종이 답안지에 씁니다.** 노션 실습 가이드를 보며 실습 0부터 7까지 순서대로 실행하세요.
- `____` 로 표시된 빈칸만 채우면 됩니다. 빈칸을 채우지 않고 실행하면 `NameError` 가 납니다.
- 페이지 번호는 **PDF 페이지 번호**(표지 = p.1)입니다. 책에 인쇄된 쪽 번호와 다릅니다.

**제출:** 종이 답안지, `학번_이름_midterm.ipynb`, `midterm_rag_results.csv`, `midterm_chat_log.csv`

## 실습 0. 데이터 준비
가이드북 PDF를 ① 근로복지공단 홈페이지 → ② GitHub 백업 순서로 내려받습니다. 둘 다 실패하면 업로드 창이 열립니다.
평가 질문 21개(`faq.csv`)는 셀 안에 들어 있어 따로 받지 않습니다.

In [ ]:
# 0. 실습 데이터 준비
import urllib.request
from pathlib import Path

DATA_DIR = Path("midterm_data")
DATA_DIR.mkdir(exist_ok=True)
PDF_PATH = DATA_DIR / "welfare_guide_2026.pdf"
CSV_PATH = DATA_DIR / "faq.csv"

PDF_URLS = [
    "https://welfare.comwel.or.kr/mobile/board/bbs.do?mCode=F010000000"
    "&cfgIdx=1&idxId=472251&op=download"
    "&fileNm=2026+%EB%B3%B5%EC%A7%80%EC%82%AC%EC%97%85"
    "%EA%B0%80%EC%9D%B4%EB%93%9C%EB%B6%81V2.pdf",
    "https://raw.githubusercontent.com/itlects/2026-1_ai-convergence-apply"
    "/main/week08_midterm/data/welfare_guide_2026.pdf",
]


def download_pdf(path, urls):
    if path.exists() and path.read_bytes()[:4] == b"%PDF":
        print("이미 준비됨:", path)
        return True
    for url in urls:
        try:
            request = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=60) as response:
                path.write_bytes(response.read())
            if path.read_bytes()[:4] == b"%PDF":
                print("다운로드 완료 ←", url[:45], "...")
                return True
        except Exception as error:
            print("실패, 다음 방법 시도:", type(error).__name__)
        path.unlink(missing_ok=True)
    return False


if not download_pdf(PDF_PATH, PDF_URLS):
    print("PDF를 직접 업로드하세요 (2026 복지사업 가이드북)")
    from google.colab import files
    uploaded = files.upload()
    PDF_PATH.write_bytes(next(iter(uploaded.values())))

CSV_PATH.write_text("""id,question,answer_page,in_document
Q01,근로자생활안정자금 융자의 금리는 얼마인가요?,4,Y
Q02,이차보전 대출은 조기상환 수수료가 있나요?,5,Y
Q03,이차보전 대출 추천서의 유효기간은 며칠인가요?,6,Y
Q04,직업훈련 생계비 대부를 받으려면 훈련이 몇 시간 이상이어야 하나요?,7,Y
Q05,체불근로자 생계비 융자 한도는 1인당 얼마인가요?,8,Y
Q06,산재근로자 차량구입비 융자 한도는 얼마인가요?,9,Y
Q07,산재근로자 융자 상환방법은 어떻게 선택하나요?,10,Y
Q08,보증료율과 보증기간별 총 보증료를 알려주세요,11,Y
Q09,근로자지원프로그램 개인상담은 1년에 몇 회기까지 무료인가요?,12,Y
Q10,휴양콘도 주말 이용은 언제까지 신청해야 하나요?,13,Y
Q11,노동자 문화예술제 대통령상 상금은 얼마인가요?,14,Y
Q12,퇴직연금 운용관리 수수료는 얼마인가요?,16,Y
Q13,근로복지공단 어린이집 2세 월 보육료는 얼마인가요?,18,Y
Q14,도산대지급금은 최대 얼마까지 지급되나요?,21,Y
Q15,간이대지급금 지급금액 상한액은 얼마인가요?,22,Y
Q16,체불청산지원 사업주 융자의 금리는 얼마인가요?,23,Y
Q17,실업급여 수급 기간은 얼마인가요?,0,N
Q18,국민연금 수령 나이는 몇 살인가요?,0,N
Q19,오늘 날씨 알려줘,0,N
Q20,해외여행 비자 발급 절차는?,0,N
Q21,주택청약 1순위 조건은?,0,N
""", encoding="utf-8")
print("PDF 크기:", PDF_PATH.stat().st_size // 1024, "KB / 평가 질문 준비 완료")

In [ ]:
# pypdf·gradio 설치 (1~2분)
!pip -q install pypdf gradio

## 실습 1. PDF 텍스트 추출 → 답안지 1-1, 1-2
- 빈칸에 `1` 을 넣어 1페이지부터 번호를 붙이세요. 페이지 번호는 답변의 **출처**로 쓰입니다.
- 출력에서 **전체 페이지 수**, **텍스트가 있는 페이지 수**, 그리고 빠진 페이지 번호를 확인하세요.

In [ ]:
# 1. PDF 텍스트 추출: 페이지 번호를 메타데이터로 함께 저장합니다.
import re
from pypdf import PdfReader

reader = PdfReader(str(PDF_PATH))
print("전체 페이지 수:", len(reader.pages))

records = []
for page_no, page in enumerate(reader.pages, start=____):  # TODO: 1
    text = page.extract_text() or ""
    text = re.sub(r"\s+", " ", text).strip()  # 줄바꿈·연속 공백 정리
    if text:
        records.append({"source": PDF_PATH.name,
                        "page": page_no,
                        "text": text})

print("텍스트가 있는 페이지 수:", len(records))
pages_with_text = {r["page"] for r in records}
empty = [n for n in range(1, len(reader.pages) + 1)
         if n not in pages_with_text]
print("텍스트가 없는 페이지:", empty)
for record in records[:5]:
    print(f"p.{record['page']:>2} | 단어 {len(record['text'].split()):>3}개 | "
          f"{record['text'][:40]}...")

## 실습 2. 청킹 (size / overlap) → 답안지 2-1, 2-2
- `step = size - ____` 빈칸에 `overlap` 을 넣으세요. 이웃 청크가 overlap개 단어를 공유합니다.
- `CHUNK_SIZE` 에 `60`, `OVERLAP` 에 `15` 를 넣으세요.

In [ ]:
def chunk_words(text, size=60, overlap=15):
    """공백 기준 단어(어절)로 나누고, 이웃 청크가 overlap개를 공유합니다."""
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("0 <= overlap < size 조건이 필요합니다.")
    words = text.split()
    step = size - ____  # TODO: overlap
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):  # 마지막 단어까지 담았으면 종료
            break
    return chunks


def build_chunks(records, size=60, overlap=15):
    rows = []
    for record in records:
        parts = chunk_words(record["text"], size, overlap)
        for number, part in enumerate(parts, start=1):
            rows.append({"source": record["source"],
                         "page": record["page"],
                         "chunk": number,
                         "text": part})
    return rows


# 함수 동작 확인 → 답안지 2-1
sample = "하나 둘 셋 넷 다섯 여섯 일곱 여덟 아홉 열"
for part in chunk_words(sample, size=4, overlap=1):
    print(part)

In [ ]:
import pandas as pd

CHUNK_SIZE = ____  # TODO: 60
OVERLAP = ____     # TODO: 15

chunks = build_chunks(records, CHUNK_SIZE, OVERLAP)
print("청크 수:", len(chunks))
chunk_table = pd.DataFrame(chunks)
chunk_table["words"] = chunk_table["text"].str.split().str.len()
chunk_table[["page", "chunk", "words", "text"]].head(5)

## 실습 3. TF-IDF 벡터화와 코사인 유사도 검색 → 답안지 3-1, 3-2, 3-3
- `analyzer="____"` 에 `"char_wb"`(어절 안 글자 조각)를 넣으세요.
- `retrieve()` 의 `[:____]` 에 `top_k` 를 넣으면 점수가 높은 순서로 top_k개를 돌려줍니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


def build_index(chunks, analyzer="char_wb", ngram_range=(2, 3)):
    """청크 목록으로 TF-IDF 사전을 만들고 문서 벡터를 계산합니다."""
    vectorizer = TfidfVectorizer(analyzer=analyzer,
                                 ngram_range=ngram_range,
                                 sublinear_tf=True)
    matrix = vectorizer.fit_transform([c["text"] for c in chunks])
    return {"chunks": chunks, "vectorizer": vectorizer, "matrix": matrix}


def retrieve(index, question, top_k=3):
    """질문과 가장 비슷한 청크 top_k개를 점수 순으로 돌려줍니다."""
    query = index["vectorizer"].transform([question])
    scores = cosine_similarity(query, index["matrix"]).ravel()
    order = np.argsort(scores)[::-1][:____]  # TODO: top_k
    return [{**index["chunks"][i], "score": float(scores[i])}
            for i in order]


demo = TfidfVectorizer(analyzer="____", ngram_range=(2, 3))  # TODO: char_wb
pieces = demo.build_analyzer()("복지사업을 신청합니다")
print(len(pieces), "개 조각:", pieces)

In [ ]:
index = build_index(chunks)
print("TF-IDF 행렬 크기 (청크 수, 조각 종류 수):", index["matrix"].shape)

questions = [
    "이차보전 대출 추천서의 유효기간은 며칠인가요?",
    "퇴직연금 운용관리 수수료는 얼마인가요?",
    "도산대지급금은 최대 얼마까지 지급되나요?",
]
for question in questions:
    print("\n질문:", question)
    for rank, hit in enumerate(retrieve(index, question, top_k=3), 1):
        print(f"  {rank}위 p.{hit['page']} 점수 {hit['score']:.3f} | "
              f"{hit['text'][:40]}...")

## 실습 4. 평가 질문으로 검색 성능 측정 → 답안지 4-1, 4-3
- `answer_page` 는 정답 페이지, `in_document=N` 은 **문서에 없는 질문**입니다.
- 임계값은 '문서에 있는 질문의 최저 점수'와 '문서에 없는 질문의 최고 점수'의 **평균**입니다. 빈칸에 `2` 를 넣으세요.
- 결과표는 `midterm_rag_results.csv` 로 저장됩니다.

In [ ]:
faq = pd.read_csv(CSV_PATH)
print("평가 질문 수:", len(faq))


def evaluate(records, faq, size=60, overlap=15,
             analyzer="char_wb", ngram_range=(2, 3)):
    """설정 하나로 검색을 실행하고 질문별 결과표와 요약을 돌려줍니다."""
    index = build_index(build_chunks(records, size, overlap),
                        analyzer, ngram_range)
    rows = []
    for item in faq.itertuples():
        hits = retrieve(index, item.question, top_k=3)
        pages = [hit["page"] for hit in hits]
        rows.append({"id": item.id,
                     "question": item.question,
                     "in_doc": item.in_document == "Y",
                     "answer_page": item.answer_page,
                     "top_pages": pages,
                     "top1": pages[0] == item.answer_page,
                     "top3": item.answer_page in pages,
                     "score": round(hits[0]["score"], 3)})
    table = pd.DataFrame(rows)
    inside = table[table["in_doc"]]
    outside = table[~table["in_doc"]]
    summary = {"size": size, "overlap": overlap, "analyzer": analyzer,
               "chunks": len(index["chunks"]),
               "top1_acc": round(float(inside["top1"].mean()), 3),
               "top3_acc": round(float(inside["top3"].mean()), 3),
               "min_in_doc": float(inside["score"].min()),
               "max_out_doc": float(outside["score"].max())}
    summary["threshold"] = round(
        (summary["min_in_doc"] + summary["max_out_doc"]) / ____, 2)  # TODO: 2
    return table, summary


result, summary = evaluate(records, faq, CHUNK_SIZE, OVERLAP)
print(summary)
result.to_csv("midterm_rag_results.csv", index=False, encoding="utf-8-sig")
result

## 실습 5. 설정 비교: 청크 크기와 분석 단위 → 답안지 4-2
셀을 실행해 세 가지 설정의 청크 수, Top-1·Top-3 정확도, 임계값을 비교합니다.
`min_in_doc > max_out_doc` 이어야 임계값 하나로 두 그룹을 나눌 수 있습니다.

In [ ]:
settings = [
    {"size": 60, "overlap": 15, "analyzer": "char_wb"},
    {"size": 30, "overlap": 8, "analyzer": "char_wb"},
    {"size": 60, "overlap": 15, "analyzer": "word", "ngram_range": (1, 1)},
]
summaries = [evaluate(records, faq, **s)[1] for s in settings]
pd.DataFrame(summaries)

## 실습 6. RAG 챗봇으로 질의응답 → 답안지 5-1, 5-2, 5-3
1. `THRESHOLD` 빈칸에 실습 4에서 나온 `threshold` 값을 넣고 셀을 실행하세요.
2. 아래에 채팅 창이 열리면 **답안지 문제 5의 질문 ①②③을 한 글자도 바꾸지 말고** 입력하세요. 글자가 다르면 점수(유사도)가 달라집니다.
3. 응답의 판정(근거 있음/문서에서 확인할 수 없습니다), 1·2순위 근거의 페이지와 점수, 근거 문장에서 찾은 답을 답안지에 옮겨 쓰세요.
4. 채팅 창이 열리지 않으면 바로 아래 '채팅 창 대신 쓰기' 셀에서 `ask("질문")` 으로 같은 응답을 볼 수 있습니다.
5. 질문과 응답은 자동으로 기록되며, 마지막 셀에서 `midterm_chat_log.csv` 로 저장합니다.

In [ ]:
THRESHOLD = ____  # TODO: 실습 4의 threshold 값 (예: 0.17)
NOT_FOUND = "문서에서 확인할 수 없습니다."
chat_log = []


def answer(question, top_k=3):
    """검색 모드 응답: 임계값 판정 + 근거 청크(페이지·점수) 표시"""
    question = (question or "").strip()
    if not question:
        return "질문을 입력하세요."
    hits = retrieve(index, question, top_k=top_k)
    best = hits[0]
    if best["score"] < THRESHOLD:
        reply = (f"{NOT_FOUND}\n(최고 점수 {best['score']:.3f}"
                 f" < 임계값 {THRESHOLD})")
        verdict = "근거 없음"
    else:
        lines = [f"근거 있음 · 1순위 p.{best['page']} · "
                 f"점수 {best['score']:.3f}", ""]
        for n, hit in enumerate(hits, start=1):
            lines.append(f"[근거 {n}] p.{hit['page']} · "
                         f"점수 {hit['score']:.3f}")
            lines.append(hit["text"])
            lines.append("")
        reply = "\n".join(lines).strip()
        verdict = "근거 있음"
    chat_log.append({"question": question, "verdict": verdict,
                     "top1_page": best["page"],
                     "top1_score": round(best["score"], 3)})
    return reply


def chat(message, history):
    return answer(message)


import gradio as gr

app = gr.ChatInterface(
    fn=chat,
    title="2026 복지사업 가이드북 질의응답",
    description=f"문서: {PDF_PATH.name} · 임계값 {THRESHOLD} · 검색 모드",
    examples=["직장어린이집 인건비 지원 금액은 얼마인가요?",
              "노동자 문화예술제 참가 자격은 무엇인가요?"],
)
app.launch()

### 채팅 창 대신 쓰기 (채팅 창이 열리지 않을 때만)

In [ ]:
def ask(question):
    print("질문:", question)
    print(answer(question))
    print("-" * 60)


ask("직장어린이집 인건비 지원 금액은 얼마인가요?")  # 연습용 질문

## 실습 7. LLM에 보낼 프롬프트 확인 (RAG의 G 단계) → 답안지 6-1
- RAG는 **검색(Retrieval)** 한 근거를 프롬프트에 붙여 LLM이 답을 **생성(Generation)** 하게 합니다.
- 이번 시험은 API 키 없이 진행하므로 LLM을 실제로 부르지 않고, LLM에 보낼 프롬프트만 만들어 확인합니다 (7주차 워크북 2의 `build_prompt` 와 같음).
- `top_k=____` 빈칸에 `2` 를 넣고 실행하세요. 출력된 프롬프트에서 근거의 개수와 페이지, 지시문을 확인합니다.

In [ ]:
INSTRUCTIONS = (
    "너는 복지사업 안내 도우미다. "
    "반드시 제공된 근거 안의 내용만 사용해 한국어로 3문장 이내로 답하라. "
    "답의 끝에 사용한 근거의 페이지를 (p.번호) 형식으로 적어라. "
    "근거에 답이 없으면 '문서에서 확인할 수 없습니다.'라고만 답하라."
)


def build_prompt(question, hits):
    blocks = [f"[근거 {n}] ({hit['source']} p.{hit['page']})\n{hit['text']}"
              for n, hit in enumerate(hits, start=1)]
    return "질문: " + question + "\n\n" + "\n\n".join(blocks)


question = "근로자 휴양콘도 이용료는 얼마인가요?"   # 답안지 문제 5의 질문 ①
hits = retrieve(index, question, top_k=____)        # TODO: 2
print("[지시문 instructions]\n" + INSTRUCTIONS)
print("\n[입력 input]\n" + build_prompt(question, hits))

## 마무리: 결과 파일 저장
채팅 질문을 모두 마친 뒤 실행하세요. 두 CSV 파일이 내려받아집니다.

In [ ]:
from google.colab import files

log = pd.DataFrame(chat_log).drop_duplicates()
display(log)
log.to_csv("midterm_chat_log.csv", index=False, encoding="utf-8-sig")
files.download("midterm_rag_results.csv")
files.download("midterm_chat_log.csv")

- 답안지의 **문제 7(RAG·LLM 핵심 용어와 코드 키워드)** 은 노트북 실행 없이 답안지에 바로 작성합니다.
- [파일 → 다운로드 → .ipynb 다운로드]로 `학번_이름_midterm.ipynb` 를 저장해 제출하세요.